# M39 — Physics-Derived Acoustic Concept Extraction & Validation

**Gate:** G2 (the make-or-break technical gate) · **Contributor:** Asif · **Requires:** ICBHI audio
+ the committed official split. **No model, no training, no GPU** — this is pure DSP.

## The question this answers

The new direction routes diagnosis through a bottleneck of *clinically-named acoustic concepts*.
That only means anything if the concepts measure what their names say. G2 asks exactly that:

> Do the DSP extractors reproduce ICBHI's coarse crackle/wheeze labels materially better than
> chance?

## What can and cannot be validated here

ICBHI annotates each respiratory cycle for **crackle presence** and **wheeze presence** — nothing
finer. So of the 9 concepts:

| | Concepts | Validated against |
|---|---|---|
| **Validatable** (2) | `crackle_score`, `wheeze_score` | ICBHI cycle labels — reported with AUROC + CI |
| **Proxy** (5) | `crackle_fine_ratio`, `crackle_rate_hz`, `wheeze_pitch_hz`, `rhonchi_score`, `inspiratory_fraction` | **nothing** — no ground truth exists |
| **Descriptive** (2) | `spectral_flatness`, `temporal_papr` | n/a — not claims about a clinical entity |

The proxy section below reports distributions and makes **no accuracy claim**. Promoting any proxy
requires the clinician labeling exercise in `CLINICIAN_LABELING_PACK.md` (Gate G0).

**Never write "fine crackle" for `crackle_fine_ratio` in the paper.** Write "physics-derived proxy
for fine-crackle fraction". Naming a proxy after the thing it proxies is how the ICBHI-metric
problem started.

## Two things done differently from M35

**Raw waveform, not log-mels.** M35 computed flatness/PAPR from a normalised log-mel via an
`exp(spec * 3.0)` un-log approximation. Fine for a soft penalty; not a defensible basis for a
number the paper names "spectral flatness". Also crackles are 5–15 ms events and the shared mel
hop is 10 ms, so a mel frame barely resolves one.

**True cycle duration, not the 8 s tiled clip.** Tiling a 2 s cycle to 8 s repeats every crackle
four times — `crackle_rate_hz` would be pure fiction.

## Three bugs the synthetic tests caught before this ever ran

`Asif's/engine/test_concept_extractors.py` builds signals with *constructed* acoustic ground truth
and asserts the extractors recover it. It found:

1. A pure 400 Hz tone scoring **rhonchi 0.970** — the rhonchi band (60–300 Hz) still had a
   locally-prominent leakage bin. Fixed by requiring the in-band peak to be a real share of the
   frame's global peak.
2. A brick-wall FFT band-pass ringing (Gibbs), producing a **phantom crackle ~10 ms after every
   real one** — a 50% inflation of `crackle_rate_hz`. Fixed with raised-cosine band edges.
3. Detection firing on **numerical residue at 1e-16** when the inter-event signal was near-silent
   (median/MAD → 0). Fixed with an absolute energy floor: a crackle is an explosive sound, not
   merely a statistical outlier.

Running on real audio would have surfaced none of these — it only shows the code executes, never
that a detector named "crackle" responds to crackles.

---
## Section 1 — Environment

In [ ]:
# ============================================================
# CELL 0 — ENVIRONMENT & PATHS
# ============================================================
import os, sys, platform

print("=" * 70)
print("M39 — Concept Extraction & Validation (Gate G2)")
print("=" * 70)
print(f"Python : {sys.version.split()[0]}  ({platform.platform()})")
print("No GPU needed — this notebook is pure DSP (numpy only).")

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_MOUNTED = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_MOUNTED = True
        print("Drive mounted.")
    except Exception as e:
        print(f"Drive mount skipped ({e}).")

if IN_COLAB and DRIVE_MOUNTED:
    BASE_DIR = "/content/drive/MyDrive/OWMTL/M39"
elif IN_COLAB:
    BASE_DIR = "/content/OWMTL/M39"
elif os.path.exists("/kaggle/working"):
    BASE_DIR = "/kaggle/working"
else:
    BASE_DIR = "./outputs_M39"

RESULTS_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"\nResults -> {RESULTS_DIR}")
print("=" * 70)

---
## Section 1b — Data

Needs two things: the ICBHI audio (downloaded below) and **`ICBHI_challenge_train_test.txt`**,
which is committed at `Asif's/ICBHI_challenge_train_test.txt` — upload it to `/content/`.

The notebook **refuses to run without the split file.** Earlier models silently fell back to a
`patient_id <= 111` rule that produced an 11-patient test set while labelling itself "official
60/40". That silent fallback is exactly what this refusal exists to prevent.

In [ ]:
# ============================================================
# CELL 0b — DOWNLOAD ICBHI (idempotent)
# ============================================================
import os

os.environ['KAGGLE_USERNAME'] = 'AsifM7'
os.environ['KAGGLE_KEY'] = 'PASTE_YOUR_KAGGLE_API_KEY_HERE'   # <-- replace before running
# Safer: from google.colab import userdata; os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import subprocess, sys, glob

_DL = "/content" if os.path.exists("/content") else "./data"
os.makedirs(_DL, exist_ok=True)


def _audio_dir():
    for d in glob.glob(os.path.join(_DL, "**", "audio_and_txt_files"), recursive=True):
        if glob.glob(os.path.join(d, "*.wav")):
            return d
    return None


if _audio_dir():
    print(f"ICBHI already present: {len(glob.glob(os.path.join(_audio_dir(), '*.wav')))} .wav")
else:
    if os.environ.get('KAGGLE_KEY', '') in ('', 'PASTE_YOUR_KAGGLE_API_KEY_HERE'):
        raise RuntimeError("Paste your Kaggle API key into this cell first.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    subprocess.check_call(["kaggle", "datasets", "download",
                           "-d", "vbookshelf/respiratory-sound-database", "-p", _DL, "--unzip"])
    assert _audio_dir(), "Download finished but no audio found."
    print(f"Done: {len(glob.glob(os.path.join(_audio_dir(), '*.wav')))} .wav")

In [ ]:
# ============================================================
# CELL 1 — DEPENDENCIES, PATHS, CONFIG
# ============================================================
import subprocess, glob


def pip_install(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for p, i in [("librosa", None), ("soundfile", None), ("numpy", None),
             ("pandas", None), ("matplotlib", None), ("tqdm", None)]:
    pip_install(p, i)
print("Dependencies ready.\n")


def find_first(pats):
    for pat in pats:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    return None


def find_audio_dir():
    for root in ("/content", "/kaggle/input", "./data", "."):
        if os.path.isdir(root):
            for d in sorted(glob.glob(os.path.join(root, "**", "audio_and_txt_files"),
                                      recursive=True)):
                if glob.glob(os.path.join(d, "*.wav")):
                    return d
    return None


DATA_ROOT = find_audio_dir()
SPLIT_FILE = find_first([
    "/content/ICBHI_challenge_train_test.txt",
    "/content/**/ICBHI_challenge_train_test.txt",
    "/kaggle/input/**/ICBHI_challenge_train_test.txt",
    "./**/ICBHI_challenge_train_test.txt",
])

print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"SPLIT_FILE : {SPLIT_FILE}")

if not DATA_ROOT:
    raise RuntimeError("ICBHI audio not found — run Cell 0b.")
if not SPLIT_FILE:
    raise RuntimeError(
        "ICBHI_challenge_train_test.txt not found. It is committed at "
        "Asif's/ICBHI_challenge_train_test.txt -- upload it to /content/ and re-run. "
        "This notebook refuses to fall back to a patient-id rule: doing so silently is what "
        "gave M2/M3/M12/M22 an 11-patient test set mislabelled as the official 60/40 split.")

CFG = {
    "model_id": "M39",
    "contributor": "Asif",
    "sample_rate": 16000,
    "min_cycle_s": 0.15,        # below this a cycle carries no usable acoustic content
    "seed": 42,
    "data_root": DATA_ROOT,
    "split_file": SPLIT_FILE,
    "results_dir": RESULTS_DIR,
}
print("\n" + "=" * 60)
for k, v in CFG.items():
    print(f"  {k:<14}: {v}")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 2 — IMPORTS & SEED
# ============================================================
import json, time, math, random, warnings
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("Ready.")

---
## Section 2 — The concept extractors

Spliced verbatim from `Asif's/engine/concept_extractors.py` so Colab needs no repo access. **Do not
edit this cell** — edit the module and regenerate the notebook. The module is covered by
`test_concept_extractors.py` (43/43 on synthetic signals with constructed ground truth).

In [ ]:
# ============================================================
# CELL 3 — CONCEPT EXTRACTORS  (spliced from Asif's/engine/concept_extractors.py)
# ============================================================
#!/usr/bin/env python3
"""
Physics-derived acoustic concept extractors for respiratory cycles — Gate G2.

WHAT THIS IS
------------
Per-cycle, label-free DSP that produces a small vector of *clinically-named* acoustic concepts.
This is the input side of the concept bottleneck: the diagnosis head sees only these numbers, so
they have to mean what their names say.

Refactored from M35's `VectorizedAcousticPhysicsLoss`, which computed two of these (spectral
flatness, temporal PAPR) as a *training regulariser* on log-mel spectrograms. Two changes:

  * They are now **standalone per-cycle measurements**, not a loss term.
  * They are computed from the **raw waveform**, not from a normalised log-mel via M35's
    `exp(spec * 3.0)` un-log approximation. That approximation was fine for a soft penalty but is
    not a defensible basis for a number the paper names "spectral flatness". Crackles are 5–15 ms
    events; at hop_length=160 (10 ms) a mel frame barely resolves one at all, which is the other
    reason this works on the waveform.

HONESTY ABOUT VALIDATION  (read before reporting any of these)
--------------------------------------------------------------
ICBHI annotates each cycle only for **crackle presence** and **wheeze presence**. So:

  VALIDATABLE against ICBHI labels ...... crackle_score, wheeze_score
  PROXIES with no ground truth .......... crackle_fine_ratio, crackle_rate_hz, wheeze_pitch_hz,
                                          rhonchi_score, inspiratory_fraction
  DESCRIPTIVE (no label concept) ........ spectral_flatness, temporal_papr

`CONCEPT_VALIDATION` below records this per concept, and the G2 notebook reports the two groups
separately. **Never write "fine crackle" for `crackle_fine_ratio` in the paper** — write
"physics-derived proxy for fine-crackle fraction". Naming a proxy after the thing it proxies is
how the ICBHI-metric problem started. If G0 returns a clinician, these proxies gain ground truth
and can be promoted.

CLINICAL GROUNDING
------------------
Thresholds follow the CORSA/ATS descriptive conventions for adventitious sounds:

  fine crackle    short (~5 ms two-cycle duration), higher centre frequency (~650 Hz)
  coarse crackle  longer (~10-15 ms), lower centre frequency (~350 Hz)
  wheeze          continuous >=100 ms, dominant frequency ~100-1000 Hz
  rhonchus        continuous >=100 ms, low pitched (<300 Hz)

These are *adult* conventions. `Gap7` already found the physics priors degrade on pediatric
airways (SPRSound) — expected, since children's airways are smaller and resonate higher. That
fragility is a finding to report, not a bug to tune away.

Dependencies: numpy only. STFT and band-pass are implemented directly so the module runs
anywhere (no librosa/scipy version coupling) and stays unit-testable on synthetic signals.
"""
import numpy as np

# name -> (validation status, one-line meaning)
CONCEPT_VALIDATION = {
    "crackle_score":        ("validatable", "strength of transient/discontinuous events"),
    "wheeze_score":         ("validatable", "strength of continuous tonal events 100-1000 Hz"),
    "crackle_fine_ratio":   ("proxy", "fraction of detected transients that are short+high-freq"),
    "crackle_rate_hz":      ("proxy", "detected transients per second"),
    "wheeze_pitch_hz":      ("proxy", "dominant tonal frequency when tonality is present"),
    "rhonchi_score":        ("proxy", "continuous tonal energy below 300 Hz"),
    "inspiratory_fraction": ("proxy", "share of cycle energy in its first (inspiratory) half"),
    "spectral_flatness":    ("descriptive", "Wiener flatness in the wheeze band (M35)"),
    "temporal_papr":        ("descriptive", "peak-to-average power ratio of the envelope (M35)"),
}

CONCEPT_NAMES = list(CONCEPT_VALIDATION)

# Paper-safe display names. The proxies are deliberately not called by the clinical term.
CONCEPT_DISPLAY = {
    "crackle_score":        "crackle presence (DSP)",
    "wheeze_score":         "wheeze presence (DSP)",
    "crackle_fine_ratio":   "proxy: fine-crackle fraction",
    "crackle_rate_hz":      "proxy: crackle rate (Hz)",
    "wheeze_pitch_hz":      "proxy: wheeze pitch (Hz)",
    "rhonchi_score":        "proxy: rhonchi (low-pitched continuous)",
    "inspiratory_fraction": "proxy: inspiratory energy fraction",
    "spectral_flatness":    "spectral flatness (wheeze band)",
    "temporal_papr":        "temporal PAPR",
}

# --- CORSA-derived constants -------------------------------------------------
FINE_MAX_WIDTH_MS = 8.0      # fine crackles are short; coarse run longer
FINE_MIN_CENTRE_HZ = 450.0   # and sit higher in frequency
WHEEZE_BAND = (100.0, 1000.0)
RHONCHI_BAND = (60.0, 300.0)
WHEEZE_MIN_MS = 100.0        # "continuous" by clinical convention
CRACKLE_BAND = (100.0, 2000.0)


# ============================================================ primitives
def _bandpass(x, sr, lo, hi, roll=0.3):
    """Zero-phase band-pass with raised-cosine edges (no filter-design dependency).

    The edges are TAPERED, not brick-wall. Zeroing FFT bins outright is a rectangular window in
    frequency, whose time-domain response is a sinc: every transient acquires ringing sidelobes.
    On the synthetic crackle train that produced a phantom detection ~10 ms after each real one
    and inflated crackle_rate_hz by 50%. A raised-cosine transition over `roll` x edge-frequency
    suppresses it.
    """
    n = len(x)
    if n == 0:
        return x
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(n, 1.0 / sr)
    H = np.ones_like(f)

    lo_w = max(lo * roll, 1e-9)
    H[f < lo - lo_w] = 0.0
    m = (f >= lo - lo_w) & (f < lo)
    H[m] = 0.5 * (1.0 - np.cos(np.pi * (f[m] - (lo - lo_w)) / lo_w))

    hi_w = max(hi * roll, 1e-9)
    m = (f > hi) & (f <= hi + hi_w)
    H[m] = 0.5 * (1.0 + np.cos(np.pi * (f[m] - hi) / hi_w))
    H[f > hi + hi_w] = 0.0

    return np.fft.irfft(X * H, n=n)


def _frames(x, win, hop):
    """(n_frames, win) view; returns an empty array if the signal is shorter than one window."""
    if len(x) < win:
        return np.empty((0, win))
    n = 1 + (len(x) - win) // hop
    idx = np.arange(win)[None, :] + hop * np.arange(n)[:, None]
    return x[idx]


def _stft_mag(x, sr, win_ms=40.0, hop_ms=10.0):
    """Magnitude STFT. Returns (mag [n_frames, n_bins], freqs)."""
    win = max(8, int(round(sr * win_ms / 1000.0)))
    hop = max(1, int(round(sr * hop_ms / 1000.0)))
    fr = _frames(np.asarray(x, dtype=float), win, hop)
    if fr.shape[0] == 0:
        return np.zeros((0, win // 2 + 1)), np.fft.rfftfreq(win, 1.0 / sr)
    return np.abs(np.fft.rfft(fr * np.hanning(win)[None, :], axis=1)), np.fft.rfftfreq(win, 1.0 / sr)


def _envelope(x, sr, win_ms=2.0):
    """Short-window RMS envelope. 2 ms resolves a 5 ms fine crackle; a 10 ms mel hop does not."""
    win = max(4, int(round(sr * win_ms / 1000.0)))
    hop = max(1, win // 2)
    fr = _frames(np.asarray(x, dtype=float), win, hop)
    if fr.shape[0] == 0:
        return np.array([]), hop / sr
    return np.sqrt((fr ** 2).mean(axis=1)), hop / sr


def _robust_z(v):
    """Median/MAD standardisation — resists the very spikes we are hunting for."""
    if len(v) == 0:
        return v
    med = np.median(v)
    mad = np.median(np.abs(v - med))
    return (v - med) / (1.4826 * mad + 1e-12)


# ============================================================ detectors
def detect_crackles(audio, sr, z_thresh=3.5, refractory_ms=6.0, min_peak_frac=0.15):
    """Detect transient events. Returns a list of dicts: onset_s, width_ms, centre_hz.

    Crackles are discontinuous, explosive and brief. We band-pass to 100-2000 Hz, build a 2 ms
    RMS envelope, and take robustly-prominent peaks. Each peak's width is measured at half
    prominence and its centre frequency from the local spectrum, which is what separates fine
    (short, high) from coarse (long, low) per CORSA.

    `refractory_ms` merges detections closer together than a crackle can physiologically repeat.
    Without it a short high-frequency crackle splits into two counts: the RMS envelope window
    (2 ms) is close to the carrier period of a ~650 Hz fine crackle (1.5 ms), so the envelope
    still ripples at the carrier rate and crosses threshold twice. This inflated crackle_rate_hz
    by ~50% on the synthetic fine-crackle train, which is what caught it.
    """
    x = _bandpass(np.asarray(audio, dtype=float), sr, *CRACKLE_BAND)
    env, dt = _envelope(x, sr)
    if len(env) < 5:
        return []
    z = _robust_z(env)

    # A crackle is an EXPLOSIVE sound, so it must carry real energy -- not merely be a
    # statistical outlier. Without this floor the median/MAD z-score fires on numerical
    # residue whenever the inter-event signal is near-silent (MAD -> 0), which is exactly what
    # happened on the synthetic trains: 27 "crackles" detected in a 10-crackle signal.
    env_floor = min_peak_frac * float(env.max()) if env.size else 0.0

    events, i, n = [], 1, len(env)
    while i < n - 1:
        if (z[i] >= z_thresh and env[i] >= env_floor
                and z[i] >= z[i - 1] and z[i] > z[i + 1]):
            half = z[i] / 2.0
            a = i
            while a > 0 and z[a] > half:
                a -= 1
            b = i
            while b < n - 1 and z[b] > half:
                b += 1
            width_ms = (b - a) * dt * 1000.0

            c0, c1 = int(a * dt * sr), int(min(len(x), (b + 1) * dt * sr))
            centre = 0.0
            if c1 - c0 >= 16:
                seg = x[c0:c1] * np.hanning(c1 - c0)
                mag = np.abs(np.fft.rfft(seg))
                fq = np.fft.rfftfreq(len(seg), 1.0 / sr)
                if mag.sum() > 0:
                    centre = float((mag * fq).sum() / mag.sum())   # spectral centroid
            events.append({"onset_s": float(i * dt), "width_ms": float(width_ms),
                           "centre_hz": centre, "_z": float(z[i])})
            i = b + 1
        else:
            i += 1

    # Merge events inside the refractory window, keeping the most prominent of each group.
    merged = []
    for e in events:
        if merged and (e["onset_s"] - merged[-1]["onset_s"]) * 1000.0 < refractory_ms:
            if e["_z"] > merged[-1]["_z"]:
                merged[-1] = e
        else:
            merged.append(e)
    for e in merged:
        e.pop("_z", None)
    return merged


def detect_wheezes(audio, sr, band=WHEEZE_BAND, min_ms=WHEEZE_MIN_MS, prominence=3.0,
                   min_band_share=0.25):
    """Detect continuous tonal events. Returns dicts: start_s, duration_ms, freq_hz.

    A wheeze is a *sustained* narrowband peak. Per frame we find the strongest in-band bin and
    require two things: it stands out from the local in-band background (`prominence`), AND it
    is a real share of the frame's total spectral peak (`min_band_share`). A run of such frames
    at a stable frequency, lasting >= min_ms, is a wheeze.

    `min_band_share` exists because local prominence alone is not enough: for a pure 400 Hz tone
    the 60-300 Hz rhonchi band still contains a locally-dominant leakage bin, which made a clean
    wheeze register as a rhonchus. Requiring the in-band peak to be comparable to the global peak
    removes that -- caught by the synthetic-signal tests, which is what they are for.

    Note the bands overlap by clinical convention: a sustained 150 Hz tone is both "low-pitched
    wheeze" and "rhonchus", so both concepts firing on it is correct, not double counting.
    """
    mag, freqs = _stft_mag(audio, sr, win_ms=40.0, hop_ms=10.0)
    if mag.shape[0] == 0:
        return []
    sel = (freqs >= band[0]) & (freqs <= band[1])
    if not sel.any():
        return []

    hop_s = 0.010
    peak_f, tonal = np.zeros(mag.shape[0]), np.zeros(mag.shape[0], dtype=bool)
    for t in range(mag.shape[0]):
        row = mag[t, sel]
        if row.max() <= 0:
            continue
        k = int(np.argmax(row))
        peak_f[t] = freqs[sel][k]
        med = np.median(row) + 1e-12
        global_peak = mag[t].max() + 1e-12
        tonal[t] = ((row[k] / med) >= prominence) and ((row[k] / global_peak) >= min_band_share)

    out, t = [], 0
    while t < len(tonal):
        if not tonal[t]:
            t += 1
            continue
        s = t
        while (t + 1 < len(tonal) and tonal[t + 1]
               and abs(peak_f[t + 1] - peak_f[s]) <= 0.25 * max(peak_f[s], 1.0)):
            t += 1
        dur = (t - s + 1) * hop_s * 1000.0
        if dur >= min_ms:
            out.append({"start_s": float(s * hop_s), "duration_ms": float(dur),
                        "freq_hz": float(np.mean(peak_f[s:t + 1]))})
        t += 1
    return out


# ============================================================ scalar concepts
def spectral_flatness(audio, sr, band=WHEEZE_BAND):
    """Wiener flatness (geometric/arithmetic mean) in-band. Tone -> 0, noise -> 1. From M35."""
    mag, freqs = _stft_mag(audio, sr)
    if mag.shape[0] == 0:
        return 1.0
    sel = (freqs >= band[0]) & (freqs <= band[1])
    p = mag[:, sel] ** 2 + 1e-12
    if p.shape[1] == 0:
        return 1.0
    flat = np.exp(np.log(p).mean(axis=1)) / (p.mean(axis=1) + 1e-12)
    return float(np.clip(flat.mean(), 0.0, 1.0))


def temporal_papr(audio, sr):
    """Peak-to-average power ratio of the envelope. Transients push this up. From M35."""
    env, _ = _envelope(audio, sr)
    if len(env) == 0:
        return 1.0
    p = env ** 2
    return float(p.max() / (p.mean() + 1e-12))


def inspiratory_fraction(audio, sr):
    """Share of cycle energy in its first half.

    PROXY ONLY. ICBHI cycle annotations bound a whole respiratory cycle and do not mark the
    inspiration/expiration boundary, so there is no ground truth to check this against. The
    assumption — inspiration precedes expiration within an annotated cycle — is conventional but
    unverified here. A dedicated phase detector (arXiv:1903.10251) is the upgrade path.
    """
    env, _ = _envelope(audio, sr, win_ms=10.0)
    if len(env) < 2:
        return 0.5
    p = env ** 2
    tot = p.sum()
    return float(p[: len(p) // 2].sum() / tot) if tot > 0 else 0.5


# ============================================================ the vector
def extract_concepts(audio, sr):
    """Full per-cycle concept vector. Keys are exactly CONCEPT_NAMES, values are finite floats."""
    x = np.asarray(audio, dtype=float)
    if x.size == 0 or not np.any(np.isfinite(x)) or np.allclose(x, 0):
        return {k: 0.0 for k in CONCEPT_NAMES}
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    dur = max(len(x) / sr, 1e-6)

    cr = detect_crackles(x, sr)
    wz = detect_wheezes(x, sr)

    if cr:
        fine = [e for e in cr
                if e["width_ms"] <= FINE_MAX_WIDTH_MS and e["centre_hz"] >= FINE_MIN_CENTRE_HZ]
        fine_ratio = len(fine) / len(cr)
    else:
        fine_ratio = 0.0

    papr = temporal_papr(x, sr)
    # squashed so the concept is bounded in [0,1] like the others; 2.5 is M35's PAPR threshold
    crackle_score = float(np.clip(len(cr) / dur / 10.0, 0, 1) * 0.5
                          + np.clip((papr - 2.5) / 20.0, 0, 1) * 0.5)

    wheeze_ms = sum(w["duration_ms"] for w in wz)
    wheeze_score = float(np.clip(wheeze_ms / (dur * 1000.0), 0.0, 1.0))
    wheeze_pitch = float(np.mean([w["freq_hz"] for w in wz])) if wz else 0.0

    rh = detect_wheezes(x, sr, band=RHONCHI_BAND)
    rhonchi_ms = sum(w["duration_ms"] for w in rh)

    return {
        "crackle_score": crackle_score,
        "wheeze_score": wheeze_score,
        "crackle_fine_ratio": float(fine_ratio),
        "crackle_rate_hz": float(len(cr) / dur),
        "wheeze_pitch_hz": wheeze_pitch,
        "rhonchi_score": float(np.clip(rhonchi_ms / (dur * 1000.0), 0.0, 1.0)),
        "inspiratory_fraction": inspiratory_fraction(x, sr),
        "spectral_flatness": spectral_flatness(x, sr),
        "temporal_papr": float(papr),
    }


def concepts_to_vector(d):
    """Dict -> fixed-order array, so downstream matrices always line up with CONCEPT_NAMES."""
    return np.array([float(d.get(k, 0.0)) for k in CONCEPT_NAMES], dtype=np.float32)

In [ ]:
# ============================================================
# CELL 4 — SELF-CHECK: the extractors behave on known signals
# ============================================================
# Re-runs the core synthetic assertions here, so a Colab run proves the spliced copy behaves
# exactly like the tested module rather than assuming it.
_sr = 16000
_t = np.arange(_sr) / _sr


def _tone(hz):
    return np.sin(2 * np.pi * hz * _t)


def _crackles(width_ms, n=12, carrier=650.0, seed=0):
    sig = np.zeros(_sr)
    w = int(width_ms / 1000 * _sr)
    for c in np.linspace(0, _sr - w - 1, n).astype(int):
        sig[c:c + w] += np.sin(2 * np.pi * carrier * np.arange(w) / _sr) * np.hanning(w)
    return sig


_c400, _cnoise = extract_concepts(_tone(400), _sr), extract_concepts(
    np.random.RandomState(0).randn(_sr) * 0.1, _sr)
_cfine = extract_concepts(_crackles(5.0, carrier=650.0), _sr)
_ccoarse = extract_concepts(_crackles(15.0, carrier=300.0, seed=1), _sr)

_checks = [
    ("400 Hz tone -> wheeze fires", _c400["wheeze_score"] > 0.8),
    ("400 Hz tone -> pitch recovered", abs(_c400["wheeze_pitch_hz"] - 400) < 25),
    ("400 Hz tone -> rhonchi does NOT fire", _c400["rhonchi_score"] < 0.15),
    ("noise -> high flatness, no wheeze",
     _cnoise["spectral_flatness"] > 0.3 and _cnoise["wheeze_score"] < 0.2),
    ("fine train -> high fine_ratio", _cfine["crackle_fine_ratio"] > 0.6),
    ("coarse train -> low fine_ratio", _ccoarse["crackle_fine_ratio"] < 0.3),
    ("crackle rate ~ constructed 12/s", 9 <= _ccoarse["crackle_rate_hz"] <= 15),
]
for _n, _ok in _checks:
    print(f"  [{'OK ' if _ok else 'FAIL'}] {_n}")
assert all(ok for _, ok in _checks), "Spliced extractors do not match tested behaviour."
print("\nSelf-check passed — the spliced extractors behave as tested.")

---
## Section 3 — Cycles and the official split

Each ICBHI annotation line is one respiratory cycle with crackle/wheeze flags. Those flags are the
only ground truth G2 has.

The split is the official file with patients **156** and **218** reassigned wholly to train — they
have recordings on *both* sides of the published split, which violates Protocol §1's
patient-independence requirement. Cost: 12 of 381 test recordings (3.1%). See
`Asif's/audit/official_split.py`.

In [ ]:
# ============================================================
# CELL 5 — PARSE CYCLES + OFFICIAL SPLIT (patient-independent corrected)
# ============================================================
rows = []
for line in open(CFG["split_file"]):
    p = line.split()
    if len(p) >= 2 and p[1].lower() in ("train", "test"):
        rows.append((p[0].replace(".wav", ""), p[1].lower()))

by_patient = {}
for stem, sp in rows:
    by_patient.setdefault(int(stem.split("_")[0]), set()).add(sp)
LEAKING = sorted(p for p, v in by_patient.items() if len(v) > 1)
n_test_before = sum(1 for _, sp in rows if sp == "test")
SPLIT_MAP = {stem: ("train" if int(stem.split("_")[0]) in LEAKING else sp) for stem, sp in rows}
n_test_after = sum(1 for v in SPLIT_MAP.values() if v == "test")

print(f"Official split: {len(rows)} recordings, {len(by_patient)} patients")
print(f"  NOT patient-independent: patients {LEAKING} appear on both sides")
print(f"  Protocol section 1 fix -> reassigned to TRAIN; test {n_test_before} -> {n_test_after} "
      f"recordings ({(n_test_before - n_test_after) / n_test_before * 100:.1f}% moved)")


def parse_annotations(txt_path):
    out = []
    for line in open(txt_path):
        p = line.split()
        if len(p) < 4:
            continue
        try:
            s, e, cr, wh = float(p[0]), float(p[1]), int(p[2]), int(p[3])
        except ValueError:
            continue
        if e > s:
            out.append({"start": s, "end": e, "crackle": cr, "wheeze": wh})
    return out


cycles = []
for wav in sorted(glob.glob(os.path.join(CFG["data_root"], "*.wav"))):
    stem = os.path.splitext(os.path.basename(wav))[0]
    txt = os.path.join(CFG["data_root"], stem + ".txt")
    if not os.path.exists(txt):
        continue
    sp = SPLIT_MAP.get(stem)
    if sp is None:
        continue                       # not in the official split file
    pid = int(stem.split("_")[0])
    for k, c in enumerate(parse_annotations(txt)):
        cycles.append({"stem": stem, "patient_id": pid, "split": sp,
                       "cycle_idx": k, "wav_path": wav, **c})

df = pd.DataFrame(cycles)
assert len(df), "No cycles parsed."
df["label4"] = df.crackle * 1 + df.wheeze * 2      # 0 normal,1 crackle,2 wheeze,3 both
df["unit_id"] = df.stem + "__" + df.cycle_idx.astype(str)
df["duration_s"] = df.end - df.start

tr_p, te_p = set(df[df.split == "train"].patient_id), set(df[df.split == "test"].patient_id)
assert not (tr_p & te_p), f"PATIENT LEAKAGE: {sorted(tr_p & te_p)[:5]}"

print(f"\nCycles: {len(df)}")
print(f"  train {int((df.split == 'train').sum()):>5} from {len(tr_p)} patients")
print(f"  test  {int((df.split == 'test').sum()):>5} from {len(te_p)} patients")
print("[OK] patient-independent verified\n")
print("Label distribution (all cycles):")
for i, nm in enumerate(["Normal", "Crackle", "Wheeze", "Both"]):
    n = int((df.label4 == i).sum())
    print(f"  {nm:<9}{n:>6}  ({n / len(df) * 100:5.1f}%)")
print(f"\ncrackle-positive: {int(df.crackle.sum())}   wheeze-positive: {int(df.wheeze.sum())}")
print(f"cycle duration: median {df.duration_s.median():.2f}s  "
      f"range {df.duration_s.min():.2f}-{df.duration_s.max():.2f}s")

In [ ]:
# ============================================================
# CELL 6 — EXTRACT CONCEPTS FOR EVERY CYCLE
# ============================================================
# Audio is loaded at the cycle's TRUE duration -- never tiled to a fixed 8 s. Tiling a 2 s cycle
# to 8 s would repeat each crackle ~4x and make crackle_rate_hz fiction.

_cache = {}


def load_full(path, sr):
    if path not in _cache:
        if len(_cache) > 40:                     # bounded: recordings are large
            _cache.clear()
        _cache[path] = librosa.load(path, sr=sr, mono=True)[0]
    return _cache[path]


recs, skipped = [], 0
t0 = time.time()
for _, r in tqdm(df.iterrows(), total=len(df), desc="extracting concepts"):
    try:
        y = load_full(r["wav_path"], CFG["sample_rate"])
        a, b = int(r["start"] * CFG["sample_rate"]), int(r["end"] * CFG["sample_rate"])
        seg = y[max(0, a):min(len(y), b)]
        if len(seg) < CFG["min_cycle_s"] * CFG["sample_rate"]:
            skipped += 1
            recs.append({k: np.nan for k in CONCEPT_NAMES})
            continue
        recs.append(extract_concepts(seg, CFG["sample_rate"]))
    except Exception:
        skipped += 1
        recs.append({k: np.nan for k in CONCEPT_NAMES})

EXTRACT_TIME = time.time() - t0
C = pd.DataFrame(recs, index=df.index)
df = pd.concat([df, C], axis=1)
_cache.clear()

print(f"\nExtracted in {EXTRACT_TIME:.0f}s ({EXTRACT_TIME / len(df) * 1000:.1f} ms/cycle)")
print(f"Skipped (too short / unreadable): {skipped} ({skipped / len(df) * 100:.2f}%)")
usable = df[CONCEPT_NAMES].notna().all(axis=1)
df = df[usable].reset_index(drop=True)
print(f"Usable cycles: {len(df)}")
print("\nConcept summary (all usable cycles):")
print(df[CONCEPT_NAMES].describe().T[["mean", "std", "min", "50%", "max"]].round(4).to_string())

---
## Section 4 — Gate G2: the validation that decides the plan

Only `crackle_score` and `wheeze_score` are tested — they are the only two with ICBHI ground truth.
Each gets an AUROC with a DeLong 95% CI on the **test split** (train-split figures are printed for
reference but the gate reads the test split).

**Gate rule:** a concept passes if its 95% CI excludes chance, i.e. it is *materially* better than
a coin flip, not merely above 0.5 by a point estimate.

In [ ]:
# ============================================================
# CELL 7 — AUROC + DeLong CI  (self-contained; mirrors Asif's/Statistics/owmtl_scores.py)
# ============================================================
def _midrank(x):
    J = np.argsort(x, kind="mergesort")
    Z = np.asarray(x, dtype=float)[J]
    N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(N); out[J] = T
    return out


def _fast_delong(mat, n_pos):
    m, k = n_pos, mat.shape[0]
    n = mat.shape[1] - m
    tx = np.vstack([_midrank(mat[r, :m]) for r in range(k)])
    ty = np.vstack([_midrank(mat[r, m:]) for r in range(k)])
    tz = np.vstack([_midrank(mat[r]) for r in range(k)])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    return aucs, np.atleast_2d(np.cov(v01, ddof=1)) / m + np.atleast_2d(np.cov(v10, ddof=1)) / n


def auroc_ci(y, s, alpha=0.05):
    y = np.asarray(y).astype(int); s = np.asarray(s, dtype=float)
    if len(set(y.tolist())) < 2:
        return None
    order = np.argsort(-y, kind="mergesort")
    a, cov = _fast_delong(s[order][None, :], int(y.sum()))
    se = math.sqrt(max(float(cov[0, 0]), 0.0))
    z = 1.959963984540054
    lo, hi = max(0.0, float(a[0]) - z * se), min(1.0, float(a[0]) + z * se)
    return {"auroc": float(a[0]), "se": se, "ci_lo": lo, "ci_hi": hi,
            "excludes_chance": bool(lo > 0.5 or hi < 0.5),
            "n_pos": int(y.sum()), "n_neg": int((y == 0).sum())}


VALIDATABLE = [(k, v) for k, (s, v) in CONCEPT_VALIDATION.items() if s == "validatable"]
TARGETS = {"crackle_score": "crackle", "wheeze_score": "wheeze"}

G2 = {}
for split in ("test", "train"):
    sub = df[df.split == split]
    print(f"\n{'=' * 74}\n{split.upper()} SPLIT  (n={len(sub)} cycles)\n{'=' * 74}")
    for concept, target in TARGETS.items():
        r = auroc_ci(sub[target].values, sub[concept].values)
        if r is None:
            continue
        mark = "PASS" if r["excludes_chance"] and r["auroc"] > 0.5 else "FAIL"
        print(f"  {concept:<16} vs ICBHI '{target}'   AUROC {r['auroc']:.4f}  "
              f"95% CI [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]   "
              f"n+={r['n_pos']} n-={r['n_neg']}   [{mark}]")
        if split == "test":
            G2[concept] = {**r, "target": target,
                           "passes": bool(r["excludes_chance"] and r["auroc"] > 0.5)}

In [ ]:
# ============================================================
# CELL 8 — THE G2 VERDICT
# ============================================================
n_pass = sum(1 for v in G2.values() if v["passes"])
print("=" * 74)
print("GATE G2 — CONCEPT-EXTRACTOR VALIDITY")
print("=" * 74)
print('Rule: a concept passes if its 95% CI excludes chance on the TEST split.\n')
for c, v in G2.items():
    print(f"  {c:<16} AUROC {v['auroc']:.4f}  [{v['ci_lo']:.4f}, {v['ci_hi']:.4f}]  "
          f"{'PASS' if v['passes'] else 'FAIL'}")

print("-" * 74)
if n_pass == 2:
    VERDICT = "PASS"
    print("  VERDICT: PASS — both validatable extractors beat chance with CIs excluding 0.5.")
    print("  Proceed to the bottleneck head (M13 re-wire). The full plan is live.")
elif n_pass == 1:
    VERDICT = "PARTIAL"
    weak = [c for c, v in G2.items() if not v["passes"]]
    print(f"  VERDICT: PARTIAL — {weak} did not clear chance.")
    print("  Per roadmap G2 'if not satisfied (a)': shrink to the minimal reliable concept set and")
    print("  proceed with a thinner bottleneck. Do NOT tune thresholds against the test split to")
    print("  rescue the failing one -- that would be fitting the gate.")
else:
    VERDICT = "FAIL"
    print("  VERDICT: FAIL — neither extractor beats chance.")
    print("  Per roadmap G2 'if not satisfied (b)': pivot to the reliability-only paper")
    print("  (physics-fragility + honest operating point + optional FM probing), which does not")
    print("  depend on clean concepts. This is a graceful degradation, not a dead end.")
print("=" * 74)

print("\nNOTE ON THE OTHER 7 CONCEPTS")
print("-" * 74)
for k, (status, meaning) in CONCEPT_VALIDATION.items():
    if status != "validatable":
        print(f"  {k:<22} [{status:<11}] {meaning}")
print("\n  These have NO ground truth in ICBHI and are NOT validated by this notebook.")
print("  Report them as proxies. Promoting them requires the clinician labeling exercise")
print("  in CLINICIAN_LABELING_PACK.md (Gate G0).")

---
## Section 5 — Proxy concepts: description only, no accuracy claim

These plots exist so the proxies can be sanity-checked by eye and reported honestly. **None of
this is validation.** A proxy behaving plausibly across label groups is face validity, which is
weaker evidence than it looks — the point of the clinician exercise is to replace it.

In [ ]:
# ============================================================
# CELL 9 — PROXY BEHAVIOUR ACROSS ICBHI LABEL GROUPS (descriptive)
# ============================================================
groups = {"Normal": (df.crackle == 0) & (df.wheeze == 0),
          "Crackle": (df.crackle == 1) & (df.wheeze == 0),
          "Wheeze": (df.crackle == 0) & (df.wheeze == 1),
          "Both": (df.crackle == 1) & (df.wheeze == 1)}

print(f"{'concept':<22}" + "".join(f"{g:>12}" for g in groups) + "   status")
print("-" * 86)
for c in CONCEPT_NAMES:
    line = f"{c:<22}"
    for g, m in groups.items():
        line += f"{df.loc[m, c].median():>12.3f}"
    print(line + f"   {CONCEPT_VALIDATION[c][0]}")
print("\n(medians; descriptive only — no test is implied by this table)")

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, c in zip(axes.ravel(), CONCEPT_NAMES):
    data = [df.loc[m, c].dropna().values for m in groups.values()]
    # positional tick labels: matplotlib renamed boxplot(labels=) -> tick_labels= in 3.9,
    # and Colab's version is not pinned. set_xticklabels works on every version.
    ax.boxplot(data, showfliers=False)
    ax.set_xticks(range(1, len(groups) + 1))
    ax.set_xticklabels(list(groups))
    status = CONCEPT_VALIDATION[c][0]
    ax.set_title(f"{c}\n[{status}]", fontsize=9,
                 color=("#1b7f3a" if status == "validatable" else "#c0392b"))
    ax.tick_params(labelsize=7)
fig.suptitle("Concept distributions by ICBHI label group — green = validatable, red = proxy",
             fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(CFG["results_dir"], "concept_distributions.png"), dpi=150)
plt.show()

---
## Section 6 — CC1 score dump + results JSON

In [ ]:
# ============================================================
# CELL 10 — CC1 RAW SCORE DUMP (Protocol section 1, essential #10)
# ============================================================
# Canonical schema, byte-identical to Asif's/Statistics/owmtl_scores.py::SCHEMA so the paired
# DeLong / McNemar tooling can consume it directly. Without this no paired test is possible.
import csv as _csv

_SCORE_SCHEMA = ["model_id", "split", "unit_type", "unit_id", "score_name", "score", "label"]


def dump_scores(path, model_id, unit_type, unit_ids, scores, labels,
                score_name="score", split="test", append=False):
    assert unit_type in ("patient", "cycle", "recording"), f"bad unit_type {unit_type!r}"
    unit_ids = [str(u) for u in unit_ids]
    scores = [float(x) for x in scores]
    labels = [int(x) for x in labels]
    assert len(unit_ids) == len(scores) == len(labels), "length mismatch"
    assert len(set(labels)) > 1, "need both classes"
    _exists = os.path.exists(path) and append
    with open(path, "a" if append else "w", newline="") as _f:
        _w = _csv.writer(_f)
        if not _exists:
            _w.writerow(_SCORE_SCHEMA)
        for _u, _s, _y in zip(unit_ids, scores, labels):
            _w.writerow([model_id, split, unit_type, _u, score_name, f"{_s:.10g}", _y])
    return path


SCORES_CSV = os.path.join(CFG["results_dir"], "scores_M39.csv")
first = True
for concept, target in TARGETS.items():
    for split in ("test", "train"):
        sub = df[df.split == split]
        dump_scores(SCORES_CSV, "M39", "cycle", sub.unit_id.values, sub[concept].values,
                    sub[target].values, score_name=concept, split=split, append=not first)
        first = False

print(f"[CC1] wrote {sum(1 for _ in open(SCORES_CSV)) - 1} rows -> {SCORES_CSV}")
print("     unit_type='cycle' -- these are CYCLE-level scores and the tooling will refuse to")
print("     pair them with patient-level scores from M29/M38. That guard is deliberate.")
print("\nUse: python3 \"Asif's/Statistics/owmtl_scores.py\" scores_M39.csv <other>.csv")

In [ ]:
# ============================================================
# CELL 11 — RESULTS JSON
# ============================================================
te = df[df.split == "test"]
results = {
    "meta": {
        "model_id": "M39",
        "model_name": "Physics-Derived Acoustic Concept Extraction & Validation (Gate G2)",
        "contributor": CFG["contributor"],
        "date_completed": time.strftime("%Y-%m-%d"),
        "is_augmented": False, "augmentation_method": "none",
        "notes": (
            "Gate G2 evidence for the concept-bottleneck direction. Computes 9 physics-derived "
            "acoustic concepts per ICBHI cycle by DSP (no model, no training, no labels) and "
            "validates the two that ICBHI can validate. "
            "Lineage: refactors M35's VectorizedAcousticPhysicsLoss (spectral flatness, temporal "
            "PAPR) from a training regulariser on log-mels into standalone per-cycle measurements "
            "on the RAW WAVEFORM, and adds 7 more concepts. M35's reported number is not reused "
            "(70/30 split, legacy metric). "
            "IMPORTANT: only crackle_score and wheeze_score are validatable -- ICBHI labels "
            "crackle/wheeze PRESENCE only. The other 7 are proxies with no ground truth and are "
            "reported descriptively. Do not name a proxy after the clinical entity it proxies; "
            "promoting them requires the clinician exercise in CLINICIAN_LABELING_PACK.md (G0). "
            "Concepts are computed on each cycle's TRUE duration, never tiled to 8 s -- tiling "
            "would repeat crackles and fabricate crackle_rate_hz."),
    },
    "config": {k: v for k, v in CFG.items() if k != "results_dir"},
    "environment": {
        "platform": ("Google Colab" if IN_COLAB else
                     "Kaggle" if os.path.exists("/kaggle/working") else "Local"),
        "python_version": platform.python_version(),
        "numpy_version": np.__version__,
    },
    "dataset_info": {
        "dataset": "ICBHI_2017", "data_source": "real_audio",
        "unit": "respiratory_cycle",
        "total_cycles": int(len(df)),
        "train_cycles": int((df.split == "train").sum()),
        "test_cycles": int(len(te)),
        "train_patients": int(df[df.split == "train"].patient_id.nunique()),
        "test_patients": int(te.patient_id.nunique()),
        "split_method": "official_60_40_patient_independent_corrected",
        "split_source": "official_file_patient_independent",
        "split_details": {"leaking_patients": LEAKING,
                          "test_recordings_official": n_test_before,
                          "test_recordings_used": n_test_after},
        "patient_leakage_verified": True,
        "cycles_skipped_too_short": int(skipped),
    },
    "efficiency": {
        "total_params": 0, "trainable_params": 0, "model_size_mb": 0.0,
        "training_time_total_s": 0,
        "extraction_time_total_s": round(float(EXTRACT_TIME), 1),
        "extraction_ms_per_cycle": round(float(EXTRACT_TIME / max(len(df), 1) * 1000), 3),
        "gpu_name": "none (pure DSP)",
        "inference_time_ms_per_sample": round(float(EXTRACT_TIME / max(len(df), 1) * 1000), 3),
        "note": "No model. Cost is DSP over the raw waveform.",
    },
    "best_epoch": {"epoch": None, "primary_metric": "concept_validation_auroc",
                   "primary_metric_value": (round(max(v["auroc"] for v in G2.values()), 4)
                                            if G2 else None)},
    "best_metrics": {
        "gate_g2": {
            "verdict": VERDICT,
            "rule": "a concept passes if its 95% DeLong CI excludes chance on the test split",
            "n_validatable": len(G2), "n_passing": int(n_pass),
            "per_concept": G2,
        },
        "concept_validation_status": {k: v[0] for k, v in CONCEPT_VALIDATION.items()},
        "concept_summary_test": {
            c: {"mean": round(float(te[c].mean()), 4), "std": round(float(te[c].std()), 4),
                "median": round(float(te[c].median()), 4)} for c in CONCEPT_NAMES},
        "unvalidated_proxy_warning": (
            "crackle_fine_ratio, crackle_rate_hz, wheeze_pitch_hz, rhonchi_score and "
            "inspiratory_fraction have NO ground truth in ICBHI. Any paper text must name them as "
            "physics-derived proxies, not as the clinical entities."),
    },
    "ablation": {
        "ablation_group": "concept_extraction",
        "ablation_role": "baseline",
        "baseline_model_id": None,
        "variable_changed": "physics-derived DSP concepts computed per cycle (no model)",
        "variables_held_constant": ["data_split: official_60_40_patient_independent_corrected",
                                    "sample_rate: 16000", "seed: 42"],
        "known_deviations": [
            "Concepts use each cycle's true duration, not the shared 8 s tiled clip, because "
            "tiling repeats transients and fabricates crackle_rate_hz.",
            "Computed from the raw waveform rather than the shared 128-mel spectrogram: at "
            "hop_length=160 (10 ms) a mel frame cannot resolve a 5 ms crackle.",
        ],
        "component_flags": {
            "has_sound_event_head": False, "has_disease_head": False,
            "has_cross_task_consistency": False, "has_cqkd_regularization": False,
            "has_openmax_rejection": False, "owl_stage": 0, "compression_clusters": None,
        },
        "loss_weights": {"sound_event_weight": None, "disease_weight": None,
                         "consistency_weight": None},
    },
    "training_history": [],
    "raw_scores_file": "scores_M39.csv",
}

path = os.path.join(CFG["results_dir"], "results_M39.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Wrote {path}")
print(json.dumps(results["best_metrics"]["gate_g2"], indent=2)[:1200])

df.to_csv(os.path.join(CFG["results_dir"], "concepts_M39.csv"), index=False)
print(f"Wrote per-cycle concept table ({len(df)} rows) -> concepts_M39.csv")

In [ ]:
# ============================================================
# FINAL CELL — HANDOFF
# ============================================================
import shutil
try:
    from IPython.display import display, FileLink
except ImportError:
    display = FileLink = None

files = sorted(glob.glob(os.path.join(CFG["results_dir"], "*.json")) +
               glob.glob(os.path.join(CFG["results_dir"], "*.csv")) +
               glob.glob(os.path.join(CFG["results_dir"], "*.png")))
print("=" * 60)
for f in files:
    print(f"Ready: {os.path.basename(f):<28} ({round(os.path.getsize(f) / 1024 ** 2, 2)} MB)")
    if display is not None:
        display(FileLink(f))
if files:
    b = os.path.join(BASE_DIR, "M39_bundle"); os.makedirs(b, exist_ok=True)
    for f in files:
        shutil.copy2(f, os.path.join(b, os.path.basename(f)))
    z = shutil.make_archive(os.path.join(BASE_DIR, "M39_handoff"), "zip", b)
    print(f"\nZIP: {z}")
    if display is not None:
        display(FileLink(z))
print("=" * 60)

---
### Reading the output

1. **The G2 verdict in Cell 8 is the deliverable.** PASS → build the bottleneck head. PARTIAL →
   shrink to the reliable concept subset. FAIL → pivot to the reliability-only paper. All three
   are live branches in the roadmap; none is a dead end.
2. **Do not tune thresholds to make a failing concept pass.** The extractor constants are
   clinically motivated (CORSA) and were fixed before seeing any ICBHI result. Adjusting them
   against the test split would be fitting the gate rather than passing it — and the constants
   live in `concept_extractors.py`, so any change is visible in the diff.
3. **Expect the proxy plots to look plausible.** That is face validity, not evidence. The
   clinician exercise is what turns any of them into a validated concept.
4. Commit `scores_M39.csv` — cycle-level, so the tooling will refuse to pair it with M29/M38's
   patient-level scores. That guard is intentional.